In [2]:
import pandas as pd , numpy as np



In [3]:
df = pd.read_csv('..//DATA/processed data/processed_balanced_data.csv')

In [4]:
col_to_drop = ['Unnamed: 0', 'city', 'state', 'cc_num','zip', 'first', 'last', 'street', 'trans_num']
df = df.drop(columns=col_to_drop)

In [5]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour

In [6]:
df.drop(columns=['trans_date_trans_time'], inplace=True)

In [7]:
merchant_rate = df.groupby("merchant")["is_fraud"].mean()

df["merchant_rate"] = df["merchant"].map(merchant_rate)
df.drop(columns=["merchant"], inplace=True)

In [8]:
df.drop(columns=['job'], inplace=True)

In [9]:
df['dob'] = pd.to_datetime(df['dob'])


In [10]:
df['age'] = (pd.Timestamp.now() - df['dob']).dt.days // 365
df.drop(columns=['dob'], inplace=True)

In [11]:
import numpy as np

def haversine_np(lon1, lat1, lon2, lat2):
    """
    Coordinates se kilometers me distance nikalne ka fast function
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c 
    return km

df['distance_km'] = haversine_np(df['long'], df['lat'], df['merch_long'], df['merch_lat'])

# 2. Ab un charo purane columns ko drop kar dein
geo_columns = ['lat', 'long', 'merch_lat', 'merch_long']
df = df.drop(columns=geo_columns)
    
print("cordinates are droped and distance column is created")


cordinates are droped and distance column is created


In [12]:
category_fraud_rate = df.groupby(df['category'])['is_fraud'].mean().sort_values(ascending=True)


df["categ_fraud_rate"] = df["category"].map(category_fraud_rate)
df.drop(columns=["category"], inplace=True)

In [13]:
df.drop(columns=['unix_time'], inplace=True)

In [14]:
df.head()

,amt,gender,city_pop,is_fraud,hour,merchant_rate,age,distance_km,categ_fraud_rate
0,3.78,M,3032,0,8,0.142857,62,38.148475,0.160541
1,166.03,M,13835,0,19,0.070423,59,73.759559,0.052608
2,46.02,M,970,0,5,0.128205,32,96.165888,0.070712
3,7.27,F,31394,0,20,0.265625,29,85.240398,0.275685
4,67.66,M,83,0,5,0.108527,68,30.376435,0.108593


In [16]:
df.to_csv( '..//DATA/processed data/final_data.csv', index=False)